In [ ]:
import pandas as pd
import numpy as np


In [ ]:
data=pd.read_csv("merged_data_17_24.csv.gz")

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()

In [ ]:
data = data[data["State"] != "DVC"]

In [ ]:
pop_data = {
    "State": [
        "Andaman and Nicobar Islands","Andhra Pradesh","Arunachal Pradesh","Assam","Bihar",
        "Chandigarh","Chhattisgarh","Dadra & Nagar Haveli and Daman & Diu","Delhi","Goa",
        "Gujarat","Haryana","Himachal Pradesh","Jammu & Kashmir","Jharkhand","Karnataka",
        "Kerala","Ladakh","Lakshadweep","Madhya Pradesh","Maharashtra","Manipur",
        "Meghalaya","Mizoram","Nagaland","Odisha","Puducherry","Punjab","Rajasthan",
        "Sikkim","Tamil Nadu","Telangana","Tripura","Uttar Pradesh","Uttarakhand","West Bengal"
    ],
    "Population": [
        380581,49386799,1383727,31205576,104099452,
        1055450,25545198,586956,16787941,1458545,
        60439692,25351462,6864602,12267013,32988134,61095297,
        33406061,274289,64473,72626809,112374333,2855794,
        2966889,1097206,1978502,41974218,1247953,27743338,
        68548437,610577,72147030,35193978,3673917,199812341,
        10086292,91276115
    ]
}

pop_df = pd.DataFrame(pop_data)

In [ ]:
data.rename(columns={
    "Hourly Demand Met (in MW)": "Demand"
}, inplace=True)

In [ ]:
def clean_state(x):
    return x.strip().lower().replace("&", "and")

data["State_clean"] = data["State"].apply(clean_state)
pop_df["State_clean"] = pop_df["State"].apply(clean_state)

In [ ]:
merged_data = data.merge(
    pop_df[["State_clean", "Population"]],
    on="State_clean",
    how="left"
)

In [ ]:
missing = merged_data[merged_data["Population"].isna()]["State"].unique()
print(missing)

In [ ]:
merged_data.sample(10)

In [ ]:
merged_data["Per_million_Demand"] = merged_data["Demand"]*1000000 / merged_data["Population"]

In [ ]:
merged_data.to_csv("merged_data_17_24_per_capita.csv.gz", index=False)

In [ ]:
import pandas as pd
merged_data=pd.read_csv("merged_data_17_24_per_capita.csv.gz")

In [ ]:
merged_data.sample(3)

In [ ]:
merged_data["DateTime"] = pd.to_datetime(merged_data["DateTime"])

In [ ]:
merged_data["DateTime"] = pd.to_datetime(merged_data["DateTime"])

merged_data["Hour"] = merged_data["DateTime"].dt.hour
merged_data["Month"] = merged_data["DateTime"].dt.month
merged_data["Day"] = merged_data["DateTime"].dt.day_name()
merged_data["Year"] = merged_data["DateTime"].dt.year

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.hist(
    merged_data["Per_million_Demand"],
    bins=50
)

plt.title("Distribution of Per Million Demand")
plt.xlabel("Per Million Demand")
plt.ylabel("Frequency")

plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,6))

sns.boxplot(
    y=merged_data["Per_million_Demand"]
)

plt.title("Boxplot of Per Million Demand")

plt.show()

In [ ]:
hourly_pattern = (
    merged_data
    .groupby("Hour")["Per_million_Demand"]
    .mean()
)

plt.figure(figsize=(10,6))

plt.plot(hourly_pattern)

plt.title("Average Hourly Electricity Demand")
plt.xlabel("Hour")
plt.ylabel("Per Million Demand")

plt.grid(True)

plt.show()

In [ ]:
monthly_pattern = (
    merged_data
    .groupby("Month")["Per_million_Demand"]
    .mean()
)

plt.figure(figsize=(10,6))

plt.plot(monthly_pattern)

plt.title("Average Monthly Electricity Demand")
plt.xlabel("Month")
plt.ylabel("Per Million Demand")

plt.grid(True)

plt.show()

In [ ]:
state_mean = (
    merged_data
    .groupby("State")["Per_million_Demand"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12,6))

state_mean.plot(kind="bar")

plt.title("Average Demand Across States")
plt.ylabel("Per Million Demand")

plt.show()

In [ ]:
state_std = (
    merged_data
    .groupby("State")["Per_million_Demand"]
    .std()
    .sort_values(ascending=False)
)

plt.figure(figsize=(12,6))

state_std.plot(kind="bar")

plt.title("Demand Variability Across States")
plt.ylabel("Standard Deviation")

plt.show()

In [ ]:
len(merged_data['State'].unique())

In [ ]:
regional_data = merged_data[
    merged_data["State"] != "Dadra and Nagar Haveli and Daman and Diu"
].copy()

In [ ]:
region_map = {

    # ---------------- NORTH ----------------
    "Punjab": "North",
    "Haryana": "North",
    "Delhi": "North",
    "Chandigarh": "North",
    "Himachal Pradesh": "North",
    "Jammu and Kashmir": "North",
    "Uttarakhand": "North",
    "Uttar Pradesh": "North",

    # ---------------- SOUTH ----------------
    "Tamil Nadu": "South",
    "Kerala": "South",
    "Karnataka": "South",
    "Andhra Pradesh": "South",
    "Telangana": "South",
    "Puducherry": "South",

    # ---------------- EAST ----------------
    "West Bengal": "East",
    "Bihar": "East",
    "Jharkhand": "East",
    "Odisha": "East",
    "Assam": "East",
    "Chhattisgarh": "East",

    # ---------------- WEST ----------------
    "Maharashtra": "West",
    "Gujarat": "West",
    "Goa": "West",
    "Rajasthan": "West",
    "Madhya Pradesh": "West",

    # ---------------- NORTH-EAST ----------------
    "Arunachal Pradesh": "North-East",
    "Manipur": "North-East",
    "Meghalaya": "North-East",
    "Mizoram": "North-East",
    "Nagaland": "North-East",
    "Sikkim": "North-East",
    "Tripura": "North-East"
}

In [ ]:
regional_data["Region"] = regional_data["State"].map(region_map)

In [ ]:
regional_data[
    regional_data["Region"].isna()
]["State"].unique()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})

fig, ax = plt.subplots(figsize=(11, 6))

regions = regional_data["Region"].dropna().unique()

for region in regions:
    temp = regional_data[regional_data["Region"] == region]

    hourly_curve = (
        temp.groupby("Hour")["Per_million_Demand"]
        .mean()
        .reindex(range(24))
    )

    ax.plot(
        hourly_curve.index,
        hourly_curve.values,
        label=region,
        linewidth=2.4,
        marker="o",
        markersize=4,
        alpha=0.95
    )

ax.set_title(
    "Regional Hourly Electricity Demand Patterns",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Hour of the Day")
ax.set_ylabel("Average Demand per Million Population")
ax.set_xticks(np.arange(0, 24, 2))
ax.set_xlim(0, 23)

ax.grid(
    True,
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


ax.legend(
    title="Region",
    title_fontsize=11,
    frameon=True,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5)
)

ax.text(
    0.01, -0.16,
    "Note: Curves represent average hourly demand normalized per million population.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})

fig, ax = plt.subplots(figsize=(11, 6))

region_order = ["North", "South", "East", "West", "North-East"]

month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

for region in region_order:
    temp = regional_data[regional_data["Region"] == region]

    monthly_curve = (
        temp.groupby("Month")["Per_million_Demand"]
        .mean()
        .reindex(range(1, 13))
    )

    ax.plot(
        monthly_curve.index,
        monthly_curve.values,
        label=region,
        linewidth=2.4,
        marker="o",
        markersize=4.5,
        alpha=0.95
    )


ax.set_title(
    "Regional Monthly Electricity Demand Patterns",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Month")
ax.set_ylabel("Average Demand per Million Population")


ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_xlim(1, 12)

ax.grid(
    True,
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


ax.legend(
    title="Region",
    title_fontsize=11,
    frameon=True,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5)
)


ax.text(
    0.01, -0.16,
    "Note: Curves represent average monthly demand normalized per million population.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})

region_order = ["North", "South", "East", "West", "North-East"]

fig, ax = plt.subplots(figsize=(11, 6))

sns.violinplot(
    data=regional_data,
    x="Region",
    y="Per_million_Demand",
    order=region_order,
    inner=None,
    cut=0,
    linewidth=1.2,
    ax=ax
)

sns.boxplot(
    data=regional_data,
    x="Region",
    y="Per_million_Demand",
    order=region_order,
    width=0.18,
    showcaps=True,
    boxprops={"facecolor": "white", "edgecolor": "black", "linewidth": 1.1},
    whiskerprops={"color": "black", "linewidth": 1.1},
    capprops={"color": "black", "linewidth": 1.1},
    medianprops={"color": "black", "linewidth": 1.4},
    showfliers=False,
    ax=ax
)


region_means = (
    regional_data
    .groupby("Region")["Per_million_Demand"]
    .mean()
    .reindex(region_order)
)

ax.scatter(
    range(len(region_order)),
    region_means.values,
    marker="D",
    s=45,
    color="black",
    label="Mean",
    zorder=5
)

ax.set_title(
    "Distribution of Electricity Demand Across Regions",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Region")
ax.set_ylabel("Hourly Demand per Million Population")

ax.grid(
    True,
    axis="y",
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=True,
    loc="upper right"
)

ax.text(
    0.01, -0.16,
    "Note: Violin width represents density; embedded boxplots show median and interquartile range.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()

plt.show()

In [ ]:
features = []

states = regional_data["State"].unique()

for state in states:
    df_state = merged_data[merged_data["State"] == state]

    #  Daily pattern
    hourly_profile = df_state.groupby("Hour")["Per_million_Demand"].mean()

    #  Monthly pattern
    monthly_profile = df_state.groupby("Month")["Per_million_Demand"].mean()

    #  Variability
    std_dev = df_state["Per_million_Demand"].std()
    mean_val = df_state["Per_million_Demand"].mean()
    cv = std_dev / mean_val if mean_val != 0 else 0

    #  Peak behavior
    peak_ratio = hourly_profile.max() / hourly_profile.min()

    # Seasonal strength
    seasonal_ratio = monthly_profile.max() / monthly_profile.min()

    feature_vector = np.concatenate([
        hourly_profile.values,
        monthly_profile.values,
        [std_dev, cv, peak_ratio, seasonal_ratio]
    ])

    features.append([state] + list(feature_vector))

In [ ]:
import pandas as pd

columns = (
    ["State"] +
    [f"h_{i}" for i in range(24)] +
    [f"m_{i}" for i in range(1,13)] +
    ["std_dev", "cv", "peak_ratio", "seasonal_ratio"]
)

feature_df = pd.DataFrame(features, columns=columns)

In [ ]:
feature_df

In [ ]:
import geopandas as gpd
india_map = gpd.read_file("india.json")

In [ ]:
feature_df2=feature_df.copy()

In [ ]:
india_map["State_clean"] = india_map["st_nm"]
feature_df2["State_clean"] = feature_df2["State"]

In [ ]:
map_df = feature_df2.merge(
    india_map,
    on="State_clean",
    how="left"
)

In [ ]:
map_df.shape

In [ ]:
sum(map_df["State"].isna())

In [ ]:
map_df.sample(10)

In [ ]:
neighbors = {}
map_df = gpd.GeoDataFrame(map_df, geometry='geometry')
for idx, row in map_df.iterrows():
    state = row["State_clean"]

    touching = map_df[map_df.geometry.touches(row.geometry)]["State_clean"].tolist()

    neighbors[state] = touching

In [ ]:
neighbors

In [ ]:
state_stats = (
    regional_data
    .groupby("State")["Per_million_Demand"]
    .mean()
    .reset_index()
)

state_stats.columns = ["State", "Mean_Demand"]

In [ ]:
state_stats

In [ ]:
map_df_mean = state_stats.merge(
    india_map,
    left_on="State",
    right_on="State_clean",
    how="left"
)

In [ ]:
map_df_mean[map_df_mean["geometry"].isna()]["State"]

In [ ]:
map_df_mean.info()

In [ ]:
from libpysal.weights import Queen

w_mean = Queen.from_dataframe(map_df_mean,    ids=map_df_mean["State"])

w_mean.transform = "R"

In [ ]:
print(w_mean.neighbors)

In [ ]:
print(w_mean.weights)

In [ ]:
neighbor_df = pd.DataFrame({
    "State": list(w_mean.neighbors.keys()),
    "Neighbors": [
        ", ".join(v)
        for v in w_mean.neighbors.values()
    ]
})

neighbor_df.head()

In [ ]:
from esda.moran import Moran

y_mean = map_df_mean["Mean_Demand"].values

moran_mean = Moran(y_mean, w_mean)

print("Moran's I:", moran_mean.I)
print("p-value:", moran_mean.p_sim)
print("z-score:", moran_mean.z_sim)

In [ ]:
from esda.moran import Moran
from splot.esda import moran_scatterplot
import matplotlib.pyplot as plt

fig, ax = moran_scatterplot(moran_mean)

plt.title("Moran Scatter Plot")

plt.show()

In [ ]:
map_df_mean.info()

In [ ]:
feature_df.info()

In [ ]:
X = feature_df.drop(columns=["State"])

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()

pcs = pca.fit_transform(X_scaled)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

components = np.arange(1, len(explained_variance) + 1)


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})

fig, ax = plt.subplots(figsize=(10.5, 6))


ax.plot(
    components,
    explained_variance,
    marker="o",
    markersize=5,
    linewidth=2.2,
    label="Individual explained variance"
)


ax.plot(
    components,
    cumulative_variance,
    marker="s",
    markersize=5,
    linewidth=2.2,
    linestyle="--",
    label="Cumulative explained variance"
)


ax.axhline(
    y=0.80,
    linestyle=":",
    linewidth=1.5,
    alpha=0.8
)

ax.text(
    components[-1],
    0.805,
    "80% threshold",
    ha="right",
    va="bottom",
    fontsize=10
)


ax.set_title(
    "Scree Plot for Principal Component Analysis",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance Ratio")

ax.set_xticks(components)
ax.set_ylim(0, 1.05)


ax.grid(
    True,
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(
    frameon=True,
    loc="best"
)


ax.text(
    0.01, -0.16,
    "Note: The first 2 components capture the dominant variation in the demand-feature space.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()

plt.show()

In [ ]:
feature_df["PC1"] = pcs[:, 0]
feature_df["PC2"] = pcs[:, 1]

In [ ]:
map_df_pca = feature_df.merge(
    india_map,
    left_on="State",
    right_on="State_clean",
    how="left"
)

In [ ]:
map_df_pca.shape

In [ ]:
import geopandas as gpd

map_df_pca = gpd.GeoDataFrame(
    map_df_pca,
    geometry="geometry"
)

In [ ]:
from libpysal.weights import Queen

w_pca = Queen.from_dataframe(map_df_pca,ids="State")

w_pca.transform = "R"

In [ ]:
w_pca.neighbors

In [ ]:
w_pca.weights

In [ ]:
map_df_pca.head(5)

In [ ]:
from esda.moran import Moran

y_pca = map_df_pca["PC1"].values

moran_pca = Moran(y_pca, w_pca)

print("Moran's I:", moran_pca.I)
print("p-value:", moran_pca.p_sim)
print("z-score:", moran_pca.z_sim)

In [ ]:
fig, ax = moran_scatterplot(moran_pca)

plt.title("Moran Scatter Plot")

plt.show()

## Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

pca_cols = [col for col in feature_df.columns if "PC" in col]

X_pca = feature_df[pca_cols]
sil_scores = []

k_values = range(2, 11)

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_pca)

    score = silhouette_score(X_pca, labels)

    sil_scores.append(score)

    print(
        f"k = {k}, "
        f"Silhouette Score = {score:.3f}"
    )

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    k_values,
    sil_scores,
    marker="o",
    markersize=6,
    linewidth=2.3,
    label="Silhouette score"
)

best_idx = np.argmax(sil_scores)
best_k = k_values[best_idx]
best_score = sil_scores[best_idx]

ax.scatter(
    best_k,
    best_score,
    s=90,
    zorder=5,
    label=f"Best k = {best_k}"
)

ax.axvline(
    best_k,
    linestyle="--",
    linewidth=1.3,
    alpha=0.75
)

ax.annotate(
    f"Maximum score = {best_score:.3f}",
    xy=(best_k, best_score),
    xytext=(best_k + 0.3, best_score),
    arrowprops=dict(arrowstyle="->", linewidth=1.1),
    fontsize=10
)

ax.set_title(
    "Silhouette Analysis for K-Means Clustering",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Number of Clusters")
ax.set_ylabel("Silhouette Score")

ax.set_xticks(k_values)

ax.grid(
    True,
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=True,
    loc="best"
)

ax.text(
    0.01, -0.18,
    "Note: Higher silhouette scores indicate better separation and compactness of clusters.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans


optimal_k = 3

kmeans_final = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

feature_df["Cluster"] = kmeans_final.fit_predict(X_pca)


cluster_map = {
    0: "Cluster 1",
    1: "Cluster 2",
    2: "Cluster 3"
}

feature_df["Cluster_Label"] = feature_df["Cluster"].map(cluster_map)



for cluster in sorted(feature_df["Cluster"].unique()):

    states_in_cluster = feature_df[
        feature_df["Cluster"] == cluster
    ]["State"].values

    print("\n" + "="*60)
    print(f"Cluster {cluster + 1}:")
    print(", ".join(states_in_cluster))


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})

fig, ax = plt.subplots(figsize=(12, 8))

sns.scatterplot(
    data=feature_df,
    x="PC1",
    y="PC2",
    hue="Cluster_Label",
    hue_order=["Cluster 1", "Cluster 2", "Cluster 3"],
    palette="Set2",
    s=130,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.95,
    ax=ax
)

for _, row in feature_df.iterrows():
    ax.text(
        row["PC1"] + 0.03,
        row["PC2"] + 0.03,
        row["State"],
        fontsize=8.5,
        alpha=0.9
    )

centroids = kmeans_final.cluster_centers_

for i, centroid in enumerate(centroids):
    ax.scatter(
        centroid[0],
        centroid[1],
        marker="X",
        s=250,
        edgecolor="black",
        linewidth=1.2,
        label=f"Centroid {i+1}",
        zorder=5
    )

ax.set_title(
    "K-Means Clustering of States in PCA Space",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("First Principal Component (PC1)")
ax.set_ylabel("Second Principal Component (PC2)")

ax.grid(
    True,
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

handles, labels = ax.get_legend_handles_labels()

cluster_handles = []
cluster_labels = []

for h, l in zip(handles, labels):
    if l in ["Cluster 1", "Cluster 2", "Cluster 3"]:
        cluster_handles.append(h)
        cluster_labels.append(l)

ax.legend(
    cluster_handles,
    cluster_labels,
    title="Cluster",
    title_fontsize=12,
    frameon=True,
    loc="best"
)

ax.text(
    0.01, -0.12,
    "Note: States are projected onto the first two principal components and grouped using K-means with k = 3.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()



plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt


cluster_map = feature_df[["State", "Cluster"]]

india_cluster_map = india_map.merge(
    cluster_map,
    left_on="State_clean",
    right_on="State",
    how="left"
)


india_cluster_map = gpd.GeoDataFrame(
    india_cluster_map,
    geometry="geometry"
)
india_cluster_map["Cluster"] = india_cluster_map["Cluster"].map({
    0: "Cluster 1",
    1: "Cluster 2",
    2: "Cluster 3",

})

fig, ax = plt.subplots(figsize=(14,12))

india_cluster_map.plot(
    column="Cluster",
    cmap="tab10",
    legend=True,
    edgecolor="black",
    linewidth=0.8,
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "black",
        "hatch": "///",
        "label": "No Data"
    },
    ax=ax
)

for idx, row in india_cluster_map.iterrows():

    if row["geometry"] is not None:

        centroid = row["geometry"].centroid

        ax.text(
            centroid.x,
            centroid.y,
            row["State_clean"],
            fontsize=7,
            ha="center"
        )

plt.title(
    "Behavioral Clusters of Electricity Demand Across India",
    fontsize=16
)

plt.axis("off")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import periodogram

regional_data["DateTime"] = pd.to_datetime(regional_data["DateTime"])

clustered_data = regional_data.merge(
    feature_df[["State", "Cluster"]],
    on="State",
    how="left"
)

cluster_series = (
    clustered_data
    .groupby(["Cluster", "DateTime"])["Per_million_Demand"]
    .mean()
    .reset_index()
)

fs = 1

for cluster in sorted(cluster_series["Cluster"].unique()):

    temp = cluster_series[cluster_series["Cluster"] == cluster].copy()
    temp = temp.sort_values("DateTime")

    y = temp["Per_million_Demand"].values


    from scipy.signal import detrend

    y_detrended = np.diff(y)
    frequencies, power = periodogram(y_detrended, fs=fs)


    valid = frequencies > 0
    periods = 1 / frequencies[valid]
    power = power[valid]


    mask = (periods >= 1) & (periods <= 20000)
    periods = periods[mask]
    power = power[mask]


    order = np.argsort(periods)
    periods = periods[order]
    power = power[order]



    seasonal_lines = {
        "Daily: 24 hours": 24,
        "Weekly: 168 hours": 168,
        "Monthly: 720 hours": 720,
        "Yearly: 8760 hours": 8760
    }


    fig, ax = plt.subplots(figsize=(11, 6))

    ax.vlines(
        periods,
        ymin=0,
        ymax=power,
        linewidth=0.8,
        alpha=0.75,
        label="Periodogram"
    )

    for label, period in seasonal_lines.items():
        ax.axvline(
            period,
            linestyle=":",
            linewidth=2,
            alpha=0.9,
            label=label
        )

    ax.set_xscale("log")

    ax.set_title(f"Periodogram of Cluster {cluster + 1}", fontweight="bold")
    ax.set_xlabel("Period in Hours (log scale)")
    ax.set_ylabel("Power")

    ax.set_xticks([2, 6, 12, 24, 168, 720, 8760])
    ax.set_xticklabels(["2", "6", "12", "24", "168", "720", "8760"])

    ax.grid(True, which="both", linestyle="--", linewidth=0.7, alpha=0.45)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(frameon=True, loc="best")

    plt.tight_layout()
    plt.show()

In [ ]:
cluster_map = feature_df[["State", "Cluster"]]

cluster_data = merged_data.merge(
    cluster_map,
    on="State",
    how="left"
)

cluster_data["DateTime"] = pd.to_datetime(cluster_data["DateTime"])

cluster_data["Hour"] = cluster_data["DateTime"].dt.hour
cluster_data["Month"] = cluster_data["DateTime"].dt.month
cluster_data["DayOfWeek"] = cluster_data["DateTime"].dt.day_name()
cluster_data["Weekday"] = cluster_data["DateTime"].dt.dayofweek
cluster_data["Date"] = cluster_data["DateTime"].dt.date

In [ ]:
cluster_ts = (
    cluster_data
    .groupby(["Cluster", "DateTime"])["Per_million_Demand"]
    .mean()
    .reset_index()
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

for c in sorted(cluster_data["Cluster"].dropna().unique()):
    temp = cluster_data[cluster_data["Cluster"] == c]

    hourly_curve = (
        temp.groupby("Hour")["Per_million_Demand"]
        .mean()
    )

    plt.plot(
        hourly_curve.index,
        hourly_curve.values,
        marker="o",
        linewidth=2,
        label=f"Cluster {c}"
    )

plt.title("Daily Seasonality Across Clusters")
plt.xlabel("Hour of Day")
plt.ylabel("Average Per Million Demand")
plt.xticks(range(24))
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekly_pattern = (
    cluster_data
    .groupby(["Cluster", "DayOfWeek"])["Per_million_Demand"]
    .mean()
    .reset_index()
)

weekly_pattern["DayOfWeek"] = pd.Categorical(
    weekly_pattern["DayOfWeek"],
    categories=day_order,
    ordered=True
)

weekly_pattern = weekly_pattern.sort_values(["Cluster", "DayOfWeek"])

plt.figure(figsize=(12,6))

for c in sorted(weekly_pattern["Cluster"].unique()):
    temp = weekly_pattern[weekly_pattern["Cluster"] == c]

    plt.plot(
        temp["DayOfWeek"],
        temp["Per_million_Demand"],
        marker="o",
        linewidth=2,
        label=f"Cluster {c}"
    )

plt.title("Weekly Seasonality Across Clusters")
plt.xlabel("Day of Week")
plt.ylabel("Average Per Million Demand")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np



plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})

fig, ax = plt.subplots(figsize=(11, 6))

month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

for c in sorted(cluster_data["Cluster"].dropna().unique()):
    temp = cluster_data[cluster_data["Cluster"] == c]

    monthly_curve = (
        temp.groupby("Month")["Per_million_Demand"]
        .mean()
        .reindex(range(1, 13))
    )

    ax.plot(
        monthly_curve.index,
        monthly_curve.values,
        marker="o",
        markersize=5,
        linewidth=2.4,
        alpha=0.95,
        label=f"Cluster {int(c) + 1}"
    )

ax.set_title(
    "Monthly Seasonality Across Clusters",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Month")
ax.set_ylabel("Average Demand per Million Population")

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_xlim(1, 12)

ax.grid(
    True,
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    title="Cluster",
    title_fontsize=11,
    frameon=True,
    loc="best"
)

ax.text(
    0.01, -0.16,
    "Note: Curves represent average monthly demand normalized per million population within each cluster.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()

plt.show()

In [ ]:
monthly_strength = []

for c in sorted(cluster_data["Cluster"].dropna().unique()):
    temp = cluster_data[cluster_data["Cluster"] == c]

    monthly_curve = temp.groupby("Month")["Per_million_Demand"].mean()

    monthly_ratio = monthly_curve.max() / (monthly_curve.min() + 1e-6)
    monthly_range = monthly_curve.max() - monthly_curve.min()

    monthly_strength.append({
        "Cluster": c,
        "Monthly_Min": monthly_curve.min(),
        "Monthly_Max": monthly_curve.max(),
        "Monthly_Range": monthly_range,
        "Monthly_Ratio": monthly_ratio
    })

monthly_strength_df = pd.DataFrame(monthly_strength)
monthly_strength_df

In [ ]:
import numpy as np
import pandas as pd

pca_cols = [col for col in feature_df.columns if col.startswith("PC")]

representatives = []

for c in sorted(feature_df["Cluster"].unique()):

    cluster_df = feature_df[feature_df["Cluster"] == c].copy()

    X_cluster = cluster_df[pca_cols].values

    centroid = X_cluster.mean(axis=0)

    distances = np.linalg.norm(X_cluster - centroid, axis=1)

    cluster_df["Distance_to_Centroid"] = distances

    representative_state = (
        cluster_df
        .sort_values("Distance_to_Centroid")
        .iloc[0]
    )

    representatives.append({
        "Cluster": c,
        "Representative_State": representative_state["State"],
        "Distance_to_Centroid": representative_state["Distance_to_Centroid"]
    })

representative_df = pd.DataFrame(representatives)

representative_df

In [ ]:
selected_states = []

for c in sorted(feature_df["Cluster"].unique()):

    cluster_df = feature_df[feature_df["Cluster"] == c].copy()

    X_cluster = cluster_df[pca_cols].values
    centroid = X_cluster.mean(axis=0)

    cluster_df["Distance_to_Centroid"] = np.linalg.norm(
        X_cluster - centroid,
        axis=1
    )

    closest = cluster_df.sort_values("Distance_to_Centroid").iloc[0]
    farthest = cluster_df.sort_values("Distance_to_Centroid").iloc[-1]

    selected_states.append({
        "Cluster": c,
        "Type": "Representative",
        "State": closest["State"]
    })

    selected_states.append({
        "Cluster": c,
        "Type": "Boundary/Extreme",
        "State": farthest["State"]
    })

selected_states_df = pd.DataFrame(selected_states)

selected_states_df

## Forecast For West Bengal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

state_name = "West Bengal"

merged_data["DateTime"] = pd.to_datetime(merged_data["DateTime"])

state_data = merged_data[
    merged_data["State"] == state_name
].copy()

state_data = state_data.sort_values("DateTime")

state_data = state_data[["DateTime", "Per_million_Demand"]]

state_data.head()

In [ ]:

train_end = "2022-12-31 23:00:00"
test_start = "2023-01-01 00:00:00"


train_wb_df = state_data[state_data["DateTime"] <= train_end].copy()
test_wb= state_data[state_data["DateTime"] >= test_start].copy()

In [ ]:
print("Missing values:", state_ts.isna().sum())
print("Start:", state_ts.index.min())
print("End:", state_ts.index.max())
print("Length:", len(state_ts))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})
rolling_avg = state_ts.rolling(window=24*7, min_periods=1).mean()




fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    state_ts.index,
    state_ts.values,
    linewidth=0.75,
    alpha=0.9,
    label="Hourly demand"
)

ax.plot(
    rolling_avg.index,
    rolling_avg.values,
    linewidth=2.0,
    label="7-day moving average"
)
ax.set_title(
    f"Hourly Electricity Demand Time Series: {state_name}",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Date")
ax.set_ylabel("Demand per Million Population")


ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))

ax.grid(
    True,
    which="major",
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)

ax.grid(
    True,
    which="minor",
    linestyle=":",
    linewidth=0.5,
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=True,
    loc="best"
)

ax.text(
    0.01, -0.16,
    "Note: The series represents hourly electricity demand normalized per million population.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()


plt.show()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

def electricity_model(df, datetime_col='DateTime', demand_col='Per_million_Demand'):
    """
    Classical decomposition using:
    - Linear Trend
    - Hourly averaging (daily seasonality)
    - Monthly averaging (yearly seasonality)
    - Weekend effect

    Returns:
    df with all components + residuals
    """

    df = df.copy()

    df[datetime_col] = pd.to_datetime(df[datetime_col])
    df = df.sort_values(datetime_col).reset_index(drop=True)

    df['t'] = np.arange(len(df))

    X = sm.add_constant(df['t'])
    trend_model = sm.OLS(df[demand_col], X).fit()
    df['trend'] = trend_model.predict(X)
    df['detrended'] = df[demand_col] - df['trend']
    df['hour'] = df[datetime_col].dt.hour
    daily_avg = df.groupby('hour')['detrended'].mean()
    df['daily_season'] = df['hour'].map(daily_avg)
    df['deseasoned_daily'] = df['detrended'] - df['daily_season']
    df['month'] = df[datetime_col].dt.month
    monthly_avg = df.groupby('month')['deseasoned_daily'].mean()
    df['yearly_season'] = df['month'].map(monthly_avg)
    df['residual1'] = df['deseasoned_daily'] - df['yearly_season']
    df['is_weekend'] = df[datetime_col].dt.weekday >= 5
    weekend_avg = df.groupby('is_weekend')['residual1'].mean()
    df['weekend_effect'] = df['is_weekend'].map(weekend_avg)
    df['final_residual'] = df['residual1'] - df['weekend_effect']

    df['fitted'] = (
        df['trend'] +
        df['daily_season'] +
        df['yearly_season'] +
        df['weekend_effect']
    )

    components = {
        'trend_model': trend_model,
        'daily_avg': daily_avg,
        'monthly_avg': monthly_avg,
        'weekend_avg': weekend_avg,
        'n_train': len(df)
    }

    return df, components

In [ ]:
linear_model_wb, comp_wb= electricity_model(train_wb_df)

In [ ]:
print(comp_wb['trend_model'].params)
comp_wb

In [ ]:
def predict_electricity_model(test_df, components,
                              datetime_col='DateTime',
                              demand_col='Per_million_Demand'):
    """
    Predict electricity demand on test data using fitted components
    from electricity_model().
    """

    test_df = test_df.copy()
    test_df[datetime_col] = pd.to_datetime(test_df[datetime_col])
    test_df = test_df.sort_values(datetime_col).reset_index(drop=True)


    trend_model = components["trend_model"]
    daily_avg = components["daily_avg"]
    monthly_avg = components["monthly_avg"]
    weekend_avg = components["weekend_avg"]
    n_train = components["n_train"]


    test_df["t"] = np.arange(n_train, n_train + len(test_df))
    X_test = sm.add_constant(test_df["t"])
    test_df["trend_pred"] = trend_model.predict(X_test)


    test_df["hour"] = test_df[datetime_col].dt.hour
    test_df["daily_season_pred"] = test_df["hour"].map(daily_avg)


    test_df["month"] = test_df[datetime_col].dt.month
    test_df["yearly_season_pred"] = test_df["month"].map(monthly_avg)


    test_df["is_weekend"] = test_df[datetime_col].dt.weekday >= 5
    test_df["weekend_effect_pred"] = test_df["is_weekend"].map(weekend_avg)

    test_df["prediction"] = (
        test_df["trend_pred"] +
        test_df["daily_season_pred"] +
        test_df["yearly_season_pred"] +
        test_df["weekend_effect_pred"]
    )

    return test_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


def residual_diagnostics(
    fitted_df,
    residual_col="final_residual",
    lags=100,
    ljungbox_lags=[24, 48, 72],state="West Bengal"
):
    """
    Residual diagnostics using:
    - ADF test
    - Ljung-Box test
    - ACF plot
    - PACF plot
    """

    resid = fitted_df[residual_col].dropna()

    adf_result = adfuller(resid)

    adf_summary = {
        "ADF Statistic": adf_result[0],
        "p-value": adf_result[1],
        "Used Lag": adf_result[2],
        "Number of Observations": adf_result[3],
        "Critical Value 1%": adf_result[4]["1%"],
        "Critical Value 5%": adf_result[4]["5%"],
        "Critical Value 10%": adf_result[4]["10%"]
    }

    adf_summary = pd.DataFrame(
        adf_summary.items(),
        columns=["Metric", "Value"]
    )

    ljungbox_result = acorr_ljungbox(
        resid,
        lags=ljungbox_lags,
        return_df=True
    )


    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 1.1,
        "figure.dpi": 120
    })

    fig, axes = plt.subplots(2, 1, figsize=(11, 7))

    plot_acf(
        resid,
        lags=lags,
        ax=axes[0],
        zero=False
    )
    axes[0].set_title(f"ACF Plot of Residuals for {state}", fontweight="bold")
    axes[0].grid(True, linestyle="--", alpha=0.45)
    axes[0].spines["top"].set_visible(False)
    axes[0].spines["right"].set_visible(False)

    plot_pacf(
        resid,
        lags=lags,
        ax=axes[1],
        zero=False,
        method="ywm"
    )
    axes[1].set_title(f"PACF Plot of Residuals for {state}", fontweight="bold")
    axes[1].grid(True, linestyle="--", alpha=0.45)
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)

    plt.tight_layout()

    plt.show()

    return adf_summary, ljungbox_result

In [ ]:
adf_wb, ljungbox_wb = residual_diagnostics(
    linear_model_wb,
    residual_col="final_residual",
    lags=100,
    ljungbox_lags=[24, 48, 72]
)

print("ADF Test Result")
display(adf_wb)

print("Ljung-Box Test Result")
display(ljungbox_wb)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_resid_model_wb = SARIMAX(
    linear_model_wb['final_residual'],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_resid_fit_wb = sarima_resid_model_wb.fit(disp=False)

print(sarima_resid_fit_wb.summary())

In [ ]:
sarima_residuals_wb = sarima_resid_fit_wb.resid.dropna()


plt.rcParams.update({
  "font.family": "serif",
  "font.size": 12,
  "axes.titlesize": 14,
  "axes.labelsize": 12,
  "xtick.labelsize": 10,
  "ytick.labelsize": 10,
  "axes.linewidth": 1.1,
  "figure.dpi": 120
})
lags=100
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

plot_acf(
  sarima_residuals_wb,
  lags=lags,
  ax=axes[0],
  zero=False
)
axes[0].set_title("ACF Plot of Final Residuals for West Bengal", fontweight="bold")
axes[0].grid(True, linestyle="--", alpha=0.45)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

plot_pacf(
  sarima_residuals_wb,
  lags=lags,
  ax=axes[1],
  zero=False,
  method="ywm"
)
axes[1].set_title("PACF Plot of Final Residuals for West Bengal", fontweight="bold")
axes[1].grid(True, linestyle="--", alpha=0.45)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

plt.tight_layout()

plt.show()

In [ ]:
test_pred_wb = predict_electricity_model(
    test_wb,
    comp_wb,   # components from electricity_model() fitted on train_hy
    datetime_col="DateTime",
    demand_col="Per_million_Demand"
)

In [ ]:
steps = len(test_pred_wb)

resid_forecast_wb = sarima_resid_fit_wb.forecast(steps=steps)

In [ ]:
resid_forecast_wb = pd.Series(
    resid_forecast_wb.values,
    index=test_pred_wb["DateTime"],
    name="sarima_residual_forecast"
)

In [ ]:
test_pred_wb["sarima_residual_forecast"] = resid_forecast_wb.values

In [ ]:
test_pred_wb["hybrid_prediction"] = (
    test_pred_wb["prediction"] +
    test_pred_wb["sarima_residual_forecast"]
)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mask = y_true != 0

    return np.mean(
        np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
    ) * 100


def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_val = mape(y_true, y_pred)

    return mae, rmse, mape_val

In [ ]:
mae_classical, rmse_classical, mape_classical = evaluate_model(
    test_pred_wb["Per_million_Demand"],
    test_pred_wb["prediction"]
)

In [ ]:
mae_hybrid, rmse_hybrid, mape_hybrid = evaluate_model(
    test_pred_wb["Per_million_Demand"],
    test_pred_wb["hybrid_prediction"]
)

In [ ]:
comparison_wb = pd.DataFrame({
    "Model": [
        "Classical Decomposition",
        "Classical + SARIMA Residual"
    ],
    "MAE": [
        mae_classical,
        mae_hybrid
    ],
    "RMSE": [
        rmse_classical,
        rmse_hybrid
    ],
    "MAPE": [
        mape_classical,
        mape_hybrid
    ]
})

comparison_wb

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,5))

plt.plot(
    test_pred_wb["DateTime"],
    test_pred_wb["Per_million_Demand"],
    label="Actual",
    linewidth=1
)



plt.plot(
    test_pred_wb["DateTime"],
    test_pred_wb["hybrid_prediction"],
    label="Predicted",
    linewidth=1
)

plt.title("Actual vs Predicted Electricity Demand for West Bengal")
plt.xlabel("Date")
plt.ylabel("Per Million Demand")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
state_ts = train_wb_df.set_index("DateTime")["Per_million_Demand"]


state_ts = state_ts.asfreq("H")

state_ts.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data = state_ts.dropna()

fig, ax = plt.subplots(figsize=(9, 5.5))

sns.histplot(
    data,
    bins=50,
    kde=True,
    stat="count",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.75,
    ax=ax
)

mean_val = data.mean()
median_val = data.median()

ax.axvline(mean_val, linestyle="--", linewidth=2, label=f"Mean = {mean_val:.2f}")
ax.axvline(median_val, linestyle=":", linewidth=2.2, label=f"Median = {median_val:.2f}")

ax.set_title(
    f"Distribution of Per Million Electricity Demand: {state_name}",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Demand per Million Population")
ax.set_ylabel("Frequency")

ax.grid(True, axis="y", linestyle="--", linewidth=0.7, alpha=0.45)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=True, loc="best")

plt.tight_layout()

plt.show()

In [ ]:
print("Minimum value:", state_ts.min())

In [ ]:
from scipy.stats import boxcox

y_original = state_ts.dropna()

y_boxcox, lambda_bc = boxcox(y_original)

print("Optimal Box-Cox lambda:", lambda_bc)

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    y_boxcox,
    bins=50,
    edgecolor="black"
)

plt.title(f"Histogram After Box-Cox Transformation - {state_name}")
plt.xlabel("Box-Cox Transformed Demand")
plt.ylabel("Frequency")

plt.grid(True)
plt.show()

In [ ]:
state_ts_boxcox = pd.Series(
    y_boxcox,
    index=y_original.index
)

In [ ]:
state_ts_diff = state_ts_boxcox.diff().dropna()

plt.figure(figsize=(16,5))

plt.plot(
    state_ts_diff.index,
    state_ts_diff.values,
    linewidth=0.6
)

plt.title(f"Box-Cox Transformed and Differenced Series - {state_name}")
plt.xlabel("Date")
plt.ylabel("Differenced Transformed Demand")

plt.grid(True)
plt.show()

In [ ]:
from scipy.signal import periodogram


y = state_ts_diff.values
freqs, power = periodogram(
    y,
    fs=1
)

freqs = freqs[1:]
power = power[1:]
periods = 1 / freqs

In [ ]:
periodogram_df = pd.DataFrame({
    "Frequency": freqs,
    "Period_Hours": periods,
    "Power": power
})

periodogram_df = periodogram_df[
    (periodogram_df["Period_Hours"] >= 2) &
    (periodogram_df["Period_Hours"] <= 20000)
]

top_periods = (
    periodogram_df
    .sort_values("Power", ascending=False)
    .head(10)
)

top_periods

In [ ]:
common_periods = {
    "6-hour": 6,
    "8-hour": 8,
    "12-hour": 12,
    "Daily": 24,
    "Weekly": 168,
    "Monthly approx.": 24*30,
    "Yearly approx.": 24*365
}

plt.figure(figsize=(12,6))

plt.plot(
    periodogram_df["Period_Hours"],
    periodogram_df["Power"],
    linewidth=1
)

plt.xscale("log")

for label, period in common_periods.items():
    plt.axvline(
        x=period,
        linestyle="--",
        linewidth=1
    )
    plt.text(
        period,
        periodogram_df["Power"].max()*0.8,
        label,
        rotation=90,
        verticalalignment="center"
    )

plt.xlabel("Period (Hours)")
plt.ylabel("Spectral Power")
plt.title(f"Periodogram with Candidate Seasonalities - {state_name}")
plt.grid(True)
plt.show()

In [ ]:
!pip install tbats

In [ ]:
from tbats import TBATS

selected_periods = [8, 12, 24]

estimator = TBATS(
    seasonal_periods=selected_periods,
    use_box_cox=False,
    use_trend=True,
    use_damped_trend=True,
    n_jobs=1
)

tbats_model = estimator.fit(state_ts_boxcox.values)

In [ ]:
forecast_tbats = tbats_model.forecast(
    steps=len(test_wb)
)
from scipy.special import inv_boxcox

forecast_wb = inv_boxcox(forecast_tbats, lambda_bc)
mae, rmse, mape_val = evaluate_model(
    test_wb['Per_million_Demand'],
    forecast_wb
)

print("TBATS")
print("Seasonal periods:", selected_periods)
print("MAE:", mae)
print("RMSE:", rmse)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,5))

plt.plot(
    test_wb['DateTime'],
    test_wb['Per_million_Demand'],
    label="Actual",
    linewidth=1
)

plt.plot(
    test_wb['DateTime'],
    forecast_wb,
    label="TBATS Forecast",
    linewidth=1
)

plt.title("TBATS Forecast - Haryana")
plt.xlabel("Date")
plt.ylabel("Per Million Demand")

plt.legend()
plt.grid(True)

plt.show()

### Forecast For Jammu and Kashmir

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

state_name = "Jammu and Kashmir"

merged_data["DateTime"] = pd.to_datetime(merged_data["DateTime"])

state_data_jk = merged_data[
    merged_data["State"] == state_name
].copy()

state_data_jk = state_data_jk.sort_values("DateTime")

state_data_jk = state_data_jk[["DateTime", "Per_million_Demand"]]

state_data_jk.head()

In [ ]:
state_ts_jk = state_data_jk.set_index("DateTime")["Per_million_Demand"]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})
rolling_avg_jk = state_ts_jk.rolling(window=24*7, min_periods=1).mean()




fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    state_ts_jk.index,
    state_ts_jk.values,
    linewidth=0.75,
    alpha=0.9,
    label="Hourly demand"
)

ax.plot(
    rolling_avg_jk.index,
    rolling_avg_jk.values,
    linewidth=2.0,
    label="7-day moving average"
)
ax.set_title(
    f"Hourly Electricity Demand Time Series: {state_name}",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Date")
ax.set_ylabel("Demand per Million Population")

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))


ax.grid(
    True,
    which="major",
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)

ax.grid(
    True,
    which="minor",
    linestyle=":",
    linewidth=0.5,
    alpha=0.25
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=True,
    loc="best"
)

ax.text(
    0.01, -0.16,
    "Note: The series represents hourly electricity demand normalized per million population.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()


plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    state_ts_jk.dropna(),
    bins=50,
    edgecolor="black"
)

plt.title(f"Histogram of Per Million Demand - {state_name}")
plt.xlabel("Per Million Demand")
plt.ylabel("Frequency")

plt.grid(True)
plt.show()

In [ ]:

train_end = "2022-12-31 23:00:00"
test_start = "2023-01-01 00:00:00"


train_jk_df = state_data_jk[state_data_jk["DateTime"] <= train_end].copy()
test_jk= state_data_jk[state_data_jk["DateTime"] >= test_start].copy()

In [ ]:
linear_model_jk, comp_jk= electricity_model(train_jk_df)

In [ ]:
print(comp_jk['trend_model'].params)
comp_jk

In [ ]:
adf_jk, ljungbox_jk = residual_diagnostics(
    linear_model_jk,
    residual_col="final_residual",
    lags=100,
    ljungbox_lags=[24, 48, 72]
)

print("ADF Test Result")
display(adf_jk)

print("Ljung-Box Test Result")
display(ljungbox_jk)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_resid_model_jk = SARIMAX(
    linear_model_jk['final_residual'],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_resid_fit_jk = sarima_resid_model_jk.fit(disp=False)

print(sarima_resid_fit_jk.summary())

In [ ]:
sarima_residuals_jk = sarima_resid_fit_jk.resid.dropna()


plt.rcParams.update({
  "font.family": "serif",
  "font.size": 12,
  "axes.titlesize": 14,
  "axes.labelsize": 12,
  "xtick.labelsize": 10,
  "ytick.labelsize": 10,
  "axes.linewidth": 1.1,
  "figure.dpi": 120
})
lags=100
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

plot_acf(
  sarima_residuals_jk,
  lags=lags,
  ax=axes[0],
  zero=False
)
axes[0].set_title("ACF Plot of Final Residuals for Jammu and Kashmir", fontweight="bold")
axes[0].grid(True, linestyle="--", alpha=0.45)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

plot_pacf(
  sarima_residuals_jk,
  lags=lags,
  ax=axes[1],
  zero=False,
  method="ywm"
)
axes[1].set_title("PACF Plot of Final Residuals for Jammu and Kashmir", fontweight="bold")
axes[1].grid(True, linestyle="--", alpha=0.45)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

plt.tight_layout()

plt.show()

In [ ]:
test_pred_jk = predict_electricity_model(
    test_jk,
    comp_jk,
    datetime_col="DateTime",
    demand_col="Per_million_Demand"
)

In [ ]:
steps = len(test_pred_jk)

resid_forecast_jk= sarima_resid_fit_jk.forecast(steps=steps)

In [ ]:
test_pred_jk["sarima_residual_forecast"] = resid_forecast_jk.values

In [ ]:
test_pred_jk["hybrid_prediction"] = (
    test_pred_jk["prediction"] +
    test_pred_jk["sarima_residual_forecast"]
)

In [ ]:
mae_classical, rmse_classical, mape_classical = evaluate_model(
    test_pred_jk["Per_million_Demand"],
    test_pred_jk["prediction"]
)

In [ ]:
mae_hybrid, rmse_hybrid, mape_hybrid = evaluate_model(
    test_pred_jk["Per_million_Demand"],
    test_pred_jk["hybrid_prediction"]
)

In [ ]:
comparison_jk = pd.DataFrame({
    "Model": [
        "Classical Decomposition",
        "Classical + SARIMA Residual"
    ],
    "MAE": [
        mae_classical,
        mae_hybrid
    ],
    "RMSE": [
        rmse_classical,
        rmse_hybrid
    ],
    "MAPE": [
        mape_classical,
        mape_hybrid
    ]
})

comparison_jk

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,5))

plt.plot(
    test_pred_jk["DateTime"],
    test_pred_jk["Per_million_Demand"],
    label="Actual",
    linewidth=1
)



plt.plot(
    test_pred_jk["DateTime"],
    test_pred_jk["hybrid_prediction"],
    label="Predicted",
    linewidth=1
)

plt.title("Actual vs Predicted Electricity Demand for Jammu and Kashmir")
plt.xlabel("Date")
plt.ylabel("Per Million Demand")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
state_ts_jk = train_jk_df.set_index("DateTime")["Per_million_Demand"]


from scipy.stats import boxcox

y_original_jk = state_ts_jk.dropna()

y_boxcox_jk, lambda_bc_jk = boxcox(y_original_jk)

print("Optimal Box-Cox lambda:", lambda_bc_jk)

In [ ]:
state_ts_boxcox_jk = pd.Series(
    y_boxcox_jk,
    index=y_original_jk.index
)

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    y_boxcox_jk,
    bins=50,
    edgecolor="black"
)

plt.title(f"Histogram After Box-Cox Transformation - Jammu and Kashmir")
plt.xlabel("Box-Cox Transformed Demand")
plt.ylabel("Frequency")

plt.grid(True)
plt.show()

In [ ]:
state_ts_diff_jk = state_ts_boxcox_jk.diff().dropna()

plt.figure(figsize=(16,5))

plt.plot(
    state_ts_diff_jk.index,
    state_ts_diff_jk.values,
    linewidth=0.6
)

plt.title(f"Box-Cox Transformed and Differenced Series - Jammu and Kashmir")
plt.xlabel("Date")
plt.ylabel("Differenced Transformed Demand")

plt.grid(True)
plt.show()

In [ ]:
from scipy.signal import periodogram


y_jk = state_ts_diff_jk.values


freqs, power = periodogram(
    y_jk,
    fs=1
)


freqs = freqs[1:]
power = power[1:]

periods = 1 / freqs

In [ ]:
periodogram_df_jk = pd.DataFrame({
    "Frequency": freqs,
    "Period_Hours": periods,
    "Power": power
})


periodogram_df_jk = periodogram_df_jk[
    (periodogram_df_jk["Period_Hours"] >= 2) &
    (periodogram_df_jk["Period_Hours"] <= 20000)
]

top_periods_jk = (
    periodogram_df_jk
    .sort_values("Power", ascending=False)
    .head(10)
)

top_periods_jk

In [ ]:
common_periods = {
    "4.8-hour": 4.8,
    "6-hour": 6,
    "12-hour": 12,
    "Daily": 24,
    "Weekly": 168,
    "Monthly approx.": 24*30,
    "Yearly approx.": 24*365
}

plt.figure(figsize=(12,6))

plt.plot(
    periodogram_df_jk["Period_Hours"],
    periodogram_df_jk["Power"],
    linewidth=1
)

plt.xscale("log")

for label, period in common_periods.items():
    plt.axvline(
        x=period,
        linestyle="--",
        linewidth=1
    )
    plt.text(
        period,
        periodogram_df_jk["Power"].max()*0.8,
        label,
        rotation=90,
        verticalalignment="center"
    )

plt.xlabel("Period (Hours)")
plt.ylabel("Spectral Power")

plt.title(f"Periodogram with Candidate Seasonalities - Jammu and Kashmir")

plt.grid(True)
plt.show()

In [ ]:
from tbats import TBATS

selected_periods = [4.8, 12, 24]

estimator = TBATS(
    seasonal_periods=selected_periods,
    use_box_cox=False,
    use_trend=True,
    use_damped_trend=True,
    n_jobs=1
)

tbats_model_jk = estimator.fit(state_ts_boxcox_jk.values)

In [ ]:

forecast_tbats_jk = tbats_model_jk.forecast(
    steps=len(test_jk)
)
from scipy.special import inv_boxcox

forecast_jk = inv_boxcox(forecast_tbats_jk, lambda_bc_jk)
mae, rmse, mape_val = evaluate_model(
    test_jk['Per_million_Demand'],
    forecast_jk
)

print("TBATS")
print("Seasonal periods:", selected_periods)
print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,5))

plt.plot(
    test_jk['DateTime'],
    test_jk['Per_million_Demand'],
    label="Actual",
    linewidth=1
)

plt.plot(
    test_jk['DateTime'],
    forecast_jk,
    label="TBATS Forecast",
    linewidth=1
)

plt.title("TBATS Forecast - Jammu and Kashmir")
plt.xlabel("Date")
plt.ylabel("Per Million Demand")

plt.legend()
plt.grid(True)

plt.show()

## Forecast for Haryana

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

state_name = "Haryana"

merged_data["DateTime"] = pd.to_datetime(merged_data["DateTime"])

state_data_hy = merged_data[
    merged_data["State"] == state_name
].copy()

state_data_hy = state_data_hy.sort_values("DateTime")

state_data_hy = state_data_hy[["DateTime", "Per_million_Demand"]]

state_data_hy.head()

In [ ]:
state_ts_hy = state_data_hy.set_index("DateTime")["Per_million_Demand"]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.1,
    "figure.dpi": 120
})
rolling_avg_hy = state_ts_hy.rolling(window=24*7, min_periods=1).mean()




fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    state_ts_hy.index,
    state_ts_hy.values,
    linewidth=0.75,
    alpha=0.9,
    label="Hourly demand"
)

ax.plot(
    rolling_avg_hy.index,
    rolling_avg_hy.values,
    linewidth=2.0,
    label="7-day moving average"
)
ax.set_title(
    f"Hourly Electricity Demand Time Series: {state_name}",
    pad=14,
    fontweight="bold"
)

ax.set_xlabel("Date")
ax.set_ylabel("Demand per Million Population")


ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))

ax.grid(
    True,
    which="major",
    linestyle="--",
    linewidth=0.7,
    alpha=0.45
)

ax.grid(
    True,
    which="minor",
    linestyle=":",
    linewidth=0.5,
    alpha=0.25
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=True,
    loc="best"
)

ax.text(
    0.01, -0.16,
    "Note: The series represents hourly electricity demand normalized per million population.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left"
)

plt.tight_layout()


plt.show()

In [ ]:
# Define train-test cutoff
train_end = "2022-12-31 23:00:00"
test_start = "2023-01-01 00:00:00"

# Split
train_hy_df= state_data_hy[state_data_hy["DateTime"] <= train_end].copy()
test_hy= state_data_hy[state_data_hy["DateTime"] >= test_start].copy()

In [ ]:
linear_model_hy, comp_hy= electricity_model(train_hy_df)

In [ ]:
print(comp_hy['trend_model'].params)
comp_hy

In [ ]:
adf_hy, ljungbox_hy = residual_diagnostics(
    linear_model_hy,
    residual_col="final_residual",
    lags=100,
    ljungbox_lags=[24, 48, 72],state="Haryana"
)

print("ADF Test Result")
display(adf_hy)

print("Ljung-Box Test Result")
display(ljungbox_hy)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_resid_model_hy = SARIMAX(
    linear_model_hy['final_residual'],
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_resid_fit_hy = sarima_resid_model_hy.fit(disp=False)

print(sarima_resid_fit_hy.summary())

In [ ]:
sarima_residuals_hy = sarima_resid_fit_hy.resid.dropna()


plt.rcParams.update({
  "font.family": "serif",
  "font.size": 12,
  "axes.titlesize": 14,
  "axes.labelsize": 12,
  "xtick.labelsize": 10,
  "ytick.labelsize": 10,
  "axes.linewidth": 1.1,
  "figure.dpi": 120
})
lags=100
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

plot_acf(
  sarima_residuals_hy,
  lags=lags,
  ax=axes[0],
  zero=False
)
axes[0].set_title("ACF Plot of Final Residuals for Haryana", fontweight="bold")
axes[0].grid(True, linestyle="--", alpha=0.45)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

plot_pacf(
  sarima_residuals_hy,
  lags=lags,
  ax=axes[1],
  zero=False,
  method="ywm"
)
axes[1].set_title("PACF Plot of Final Residuals for Haryana", fontweight="bold")
axes[1].grid(True, linestyle="--", alpha=0.45)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

plt.tight_layout()

plt.show()

In [ ]:
test_pred_hy = predict_electricity_model(
    test_hy,
    comp_hy,   # components from electricity_model() fitted on train_hy
    datetime_col="DateTime",
    demand_col="Per_million_Demand"
)

In [ ]:
steps = len(test_pred_hy)

resid_forecast_hy = sarima_resid_fit_hy.forecast(steps=steps)

In [ ]:
resid_forecast_hy = pd.Series(
    resid_forecast_hy.values,
    index=test_pred_hy["DateTime"],
    name="sarima_residual_forecast"
)

In [ ]:
test_pred_hy["sarima_residual_forecast"] = resid_forecast_hy.values

In [ ]:
test_pred_hy["hybrid_prediction"] = (
    test_pred_hy["prediction"] +
    test_pred_hy["sarima_residual_forecast"]
)

In [ ]:
mae_classical, rmse_classical, mape_classical = evaluate_model(
    test_pred_hy["Per_million_Demand"],
    test_pred_hy["prediction"]
)

In [ ]:
mae_hybrid, rmse_hybrid, mape_hybrid = evaluate_model(
    test_pred_hy["Per_million_Demand"],
    test_pred_hy["hybrid_prediction"]
)

In [ ]:
comparison_hy = pd.DataFrame({
    "Model": [
        "Classical Decomposition",
        "Classical + SARIMA Residual"
    ],
    "MAE": [
        mae_classical,
        mae_hybrid
    ],
    "RMSE": [
        rmse_classical,
        rmse_hybrid
    ],
    "MAPE": [
        mape_classical,
        mape_hybrid
    ]
})

comparison_hy

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,5))

plt.plot(
    test_pred_hy["DateTime"],
    test_pred_hy["Per_million_Demand"],
    label="Actual",
    linewidth=1
)



plt.plot(
    test_pred_hy["DateTime"],
    test_pred_hy["hybrid_prediction"],
    label="Predicted",
    linewidth=1
)

plt.title("Actual vs Predicted Electricity Demand for Haryana")
plt.xlabel("Date")
plt.ylabel("Per Million Demand")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
state_ts_hy = train_hy_df.set_index("DateTime")["Per_million_Demand"]

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    state_ts_hy.dropna(),
    bins=50,
    edgecolor="black"
)

plt.title(f"Histogram of Per Million Demand - Haryana")
plt.xlabel("Per Million Demand")
plt.ylabel("Frequency")

plt.grid(True)
plt.show()

In [ ]:
from scipy.stats import boxcox

y_original_hy = state_ts_hy.dropna()

y_boxcox_hy, lambda_bc_hy = boxcox(y_original_hy)

print("Optimal Box-Cox lambda:", lambda_bc_hy)

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    y_boxcox_hy,
    bins=50,
    edgecolor="black"
)

plt.title(f"Histogram After Box-Cox Transformation - Haryana")
plt.xlabel("Box-Cox Transformed Demand")
plt.ylabel("Frequency")

plt.grid(True)
plt.show()

In [ ]:
state_ts_boxcox_hy = pd.Series(
    y_boxcox_hy,
    index=y_original_hy.index
)

In [ ]:
state_ts_diff_hy = state_ts_boxcox_hy.diff().dropna()

plt.figure(figsize=(16,5))

plt.plot(
    state_ts_diff_hy.index,
    state_ts_diff_hy.values,
    linewidth=0.6
)

plt.title(f"Box-Cox Transformed and Differenced Series - Haryana")
plt.xlabel("Date")
plt.ylabel("Differenced Transformed Demand")

plt.grid(True)
plt.show()

In [ ]:
from scipy.signal import periodogram

y_hy = state_ts_diff_hy.values

freqs, power = periodogram(
    y_hy,
    fs=1
)

freqs = freqs[1:]
power = power[1:]

periods = 1 / freqs

In [ ]:
periodogram_df_hy = pd.DataFrame({
    "Frequency": freqs,
    "Period_Hours": periods,
    "Power": power
})


periodogram_df_hy = periodogram_df_hy[
    (periodogram_df_hy["Period_Hours"] >= 2) &
    (periodogram_df_hy["Period_Hours"] <= 20000)
]

top_periods_hy = (
    periodogram_df_hy
    .sort_values("Power", ascending=False)
    .head(10)
)

top_periods_hy

In [ ]:
common_periods = {
    "6-hour": 6,
    "8-hour": 8,
    "12-hour": 12,
    "Daily": 24,
    "Weekly": 168,
    "Monthly approx.": 24*30,
    "Yearly approx.": 24*365
}

plt.figure(figsize=(12,6))

plt.plot(
    periodogram_df_hy["Period_Hours"],
    periodogram_df_hy["Power"],
    linewidth=1
)

plt.xscale("log")

for label, period in common_periods.items():
    plt.axvline(
        x=period,
        linestyle="--",
        linewidth=1
    )
    plt.text(
        period,
        periodogram_df_hy["Power"].max()*0.8,
        label,
        rotation=90,
        verticalalignment="center"
    )

plt.xlabel("Period (Hours)")
plt.ylabel("Spectral Power")

plt.title(f"Periodogram with Candidate Seasonalities - Haryana")

plt.grid(True)
plt.show()

In [ ]:
from tbats import TBATS

selected_periods = [8, 12, 24]

estimator = TBATS(
    seasonal_periods=selected_periods,
    use_box_cox=False,
    use_trend=True,
    use_damped_trend=True,
    n_jobs=1
)

tbats_model_hy = estimator.fit(state_ts_boxcox_hy.values)

In [ ]:
from scipy.special import inv_boxcox

forecast_hy = inv_boxcox(forecast_tbats_hy, lambda_bc_hy)

In [ ]:
mae, rmse, mape_val = evaluate_model(
    test_hy['Per_million_Demand'],
    forecast_hy
)

print("TBATS")
print("Seasonal periods:", selected_periods)
print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,5))

plt.plot(
    test_hy['DateTime'],
    test_hy['Per_million_Demand'],
    label="Actual",
    linewidth=1
)

plt.plot(
    test_hy['DateTime'],
    forecast_hy,
    label="TBATS Forecast",
    linewidth=1
)

plt.title("TBATS Forecast - Haryana")
plt.xlabel("Date")
plt.ylabel("Per Million Demand")

plt.legend()
plt.grid(True)

plt.show()